In [23]:
import pickle
import numpy as np
import pandas as pd
import os

from nsga_population import *
from typing import List
from individual import Individual

import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
log_base = "..\\logdata_2\\"
log_folders = [
    "Experiment_2_2025-02-28_15_18_40_classical",
    "Experiment_2_2025-02-28_15_18_40_classical_2",
    "Experiment_2_2025-02-19_23_23_50_random_key_main",
    "Experiment_2_2025-02-26_08_05_08_random_key_main_2",
    "Experiment_2_2025-02-20_22_19_28_hash_base",
    "Experiment_2_2025-02-26_08_05_08_hash_base_2",
    "Experiment_2_2025-02-24_09_40_06_improved_random_key",
    "Experiment_2_2025-02-25_10_04_37_improved_random_key_2",
]

In [25]:
# NB: Make sure that all experiments to be compared have run the same amount of generations
reduced_agg_columns = ["Min Makespan", "Min Mean Flow Time", "Spread", "N Fronts"]
concat_dfs = []
for log_folder in log_folders:
    # For each log folder, create the full path
    cur_log_path = os.path.join(log_base, log_folder)
    for cur_file in os.listdir(cur_log_path):
        # For each item in the log folder, find the csv file in the root
        if cur_file[-4:] == ".csv":
            # Create the path to the csv file
            cur_csv_file_path = os.path.join(cur_log_path, cur_file)
            # Read the csv file from disc
            result_df = pd.read_csv(cur_csv_file_path, index_col=False)
            # Remove data from all iterations except the last one
            result_df = result_df[result_df["Iteration"] == result_df["Iteration"].max()].reset_index(drop=True)
            # Aggregate over the repetitions
            result_df = result_df.groupby(["Problem", "Candidate"]).agg({col_name : ["min", "mean", "std", "count"] for col_name in reduced_agg_columns})
            concat_dfs.append(result_df)

convergence_df = pd.concat(concat_dfs, axis=0)
convergence_df.rename(index={"classical": "NSGA-II", "qmea_enhanced_random_key" : "srkQMEA", "quantum_hash_base_encoding" : "rQMEA", "quantum_position_encoding_restricted":"rkQMEA"}, inplace=True)
convergence_df


Min Makespan                          Min Mean Flow Time  \
                           min    mean        std count                min   
Problem Candidate                                                            
abz7    NSGA-II          694.0   704.8   9.186947    10         574.500000   
abz8    NSGA-II          713.0   729.5   8.343327    10         596.350000   
abz9    NSGA-II          737.0   757.9   9.643075    10         585.450000   
ft06    NSGA-II           57.0    57.0   0.000000    10          39.666667   
ft10    NSGA-II          968.0   992.6  17.218853    10         644.800000   
...                        ...     ...        ...   ...                ...   
la24    srkQMEA          988.0   998.7   6.684144    10         784.600000   
la25    srkQMEA         1034.0  1050.2  11.272385    10         797.133333   
la26    srkQMEA         1229.0  1249.9   9.993887    10         942.900000   
la27    srkQMEA         1294.0  1315.3  12.184234    10         983.550000   
la29    srkQMEA         1234.0  1251.2  10.706177    10         920.550000   

                                                   Spread                      \
                          mean        std count       min      mean       std   
Problem Candidate                                                               
abz7    NSGA-II     591.570000   8.084010    10  0.951989  0.977041  0.014080   
abz8    NSGA-II     607.905000   6.816502    10  0.932374  0.971278  0.018843   
abz9    NSGA-II     593.615000   8.328133    10  0.882850       inf       NaN   
ft06    NSGA-II      39.666667   0.000000    10  0.969855  0.969855  0.000000   
ft10    NSGA-II     681.350000  16.353474    10  0.890523  0.961395  0.027389   
...                        ...        ...   ...       ...       ...       ...   
la24    srkQMEA     813.933333  14.693175    10  0.902507  0.970718  0.028472   
la25    srkQMEA     814.533333  10.977913    10  0.902831  0.947873  0.028198   
la26    srkQMEA     961.795000  12.271680    10  0.933619  0.950991  0.012514   
la27    srkQMEA    1000.165000  10.164728    10  0.918586  0.951743  0.014831   
la29    srkQMEA     945.590000  17.843530    10  0.881324  0.933714  0.037095   

                        N Fronts                        
                  count      min  mean       std count  
Problem Candidate                                       
abz7    NSGA-II      10        2   2.4  0.516398    10  
abz8    NSGA-II      10        2   2.1  0.316228    10  
abz9    NSGA-II      10        2   2.3  0.674949    10  
ft06    NSGA-II      10        2   2.0  0.000000    10  
ft10    NSGA-II      10        2   2.1  0.316228    10  
...                 ...      ...   ...       ...   ...  
la24    srkQMEA      10       11  13.4  1.837873    10  
la25    srkQMEA      10        9  10.8  1.475730    10  
la26    srkQMEA      10       10  11.2  1.398412    10  
la27    srkQMEA      10        7  10.5  1.649916    10  
la29    srkQMEA      10        8  10.4  1.264911    10  

[64 rows x 16 columns]

In [26]:
to_latx_df = convergence_df.iloc[:, [0, 1, 2, 4, 5, 6]]
to_latx_df = to_latx_df.astype({("Min Makespan", "min") : "int", ("Min Mean Flow Time", "min") : "int"})
to_latx_df = to_latx_df.sort_index()

In [27]:
cur_latex_content = to_latx_df.to_latex(index=True, formatters={"Problem":str.upper}, float_format="{:.2f}".format)
with open("Latex_convergence_table.tex", "w") as latex_file:
    latex_file.write(cur_latex_content)

In [28]:
to_latx_df

Min Makespan                    Min Mean Flow Time  \
                           min    mean        std                min   
Problem Candidate                                                      
abz7    NSGA-II            694   704.8   9.186947                574   
        rQMEA              710   715.3   3.128720                591   
        rkQMEA             702   710.7   5.498485                590   
        srkQMEA            701   711.5   5.700877                591   
abz8    NSGA-II            713   729.5   8.343327                596   
...                        ...     ...        ...                ...   
la29    srkQMEA           1234  1251.2  10.706177                920   
la40    NSGA-II           1275  1292.8  11.477514               1044   
        rQMEA             1281  1301.0   9.104334               1090   
        rkQMEA            1281  1297.7   8.857514               1085   
        srkQMEA           1287  1295.2   5.493430               1071   

                                           
                          mean        std  
Problem Candidate                          
abz7    NSGA-II     591.570000   8.084010  
        rQMEA       599.945000   5.265055  
        rkQMEA      597.115000   5.303722  
        srkQMEA     599.565000   4.797977  
abz8    NSGA-II     607.905000   6.816502  
...                        ...        ...  
la29    srkQMEA     945.590000  17.843530  
la40    NSGA-II    1080.793333  13.789581  
        rQMEA      1097.406667   6.413601  
        rkQMEA     1092.446667   4.932938  
        srkQMEA    1091.940000   8.029295  

[64 rows x 6 columns]

In [29]:
to_latx_df

Min Makespan                    Min Mean Flow Time  \
                           min    mean        std                min   
Problem Candidate                                                      
abz7    NSGA-II            694   704.8   9.186947                574   
        rQMEA              710   715.3   3.128720                591   
        rkQMEA             702   710.7   5.498485                590   
        srkQMEA            701   711.5   5.700877                591   
abz8    NSGA-II            713   729.5   8.343327                596   
...                        ...     ...        ...                ...   
la29    srkQMEA           1234  1251.2  10.706177                920   
la40    NSGA-II           1275  1292.8  11.477514               1044   
        rQMEA             1281  1301.0   9.104334               1090   
        rkQMEA            1281  1297.7   8.857514               1085   
        srkQMEA           1287  1295.2   5.493430               1071   

                                           
                          mean        std  
Problem Candidate                          
abz7    NSGA-II     591.570000   8.084010  
        rQMEA       599.945000   5.265055  
        rkQMEA      597.115000   5.303722  
        srkQMEA     599.565000   4.797977  
abz8    NSGA-II     607.905000   6.816502  
...                        ...        ...  
la29    srkQMEA     945.590000  17.843530  
la40    NSGA-II    1080.793333  13.789581  
        rQMEA      1097.406667   6.413601  
        rkQMEA     1092.446667   4.932938  
        srkQMEA    1091.940000   8.029295  

[64 rows x 6 columns]